# Discriminative Embeddings

An application of neural nets that has been growing in importance is in
learning useful *embeddings*.  An embedding is a low-dimensional vector
representation of a data point, which encodes some useful information.  One example of an embedding is PCA, where we can condense each point down to a point in some lower-dimensional space, without losing much variance. Another is a pretrained feature extractor, where we can input a high-dimensional data point like an image, and output a smaller number of meaningful features that represent that data.

These embeddings are steps towards some downstream goal, like image classification. However, more and more often, the embeddings themselves are the point. Often, the values of the output vectors themselves are meaningless; that is, if the embedding is 128-dimensional, the fact that the first feature might be high is not significant or interpretable. Instead, we seek to learn embeddings where things that are similar in the original space are close in the embeddings space.

We are going to learn two ways of defining and learning embeddings. Today is about discriminative embeddings, learned via contrastive learning.

## Contrastive Learning

Contrastive learning is a technique used to train models to produce these embeddings by ensuring that similar data points are positioned closely together and dissimilar points are pushed apart. This establishes a "push-pull" dynamic within the embedding space.

In Natural Language Processing (NLP), similarity can be defined by context. For instance, words that can fill the same gap in a sentence (e.g., "dog" or "cat" in "The `____` slept on the rug") are considered similar. By analyzing large, unlabeled corpora, models can learn these complex relationships without explicit human intervention.

## Facial Recognition and Triplet Loss

A practical application of contrastive learning is facial recognition. The objective is to ensure that various images of the same person result in similar embeddings, despite differences in lighting, angle, or accessories, while images of different people remain distinct.

### Training with Triplets

Training utilizes a neural network $E$ (E is for *encoder*) to map an image $x$ to an embedding $E(x)$. The model processes "triplets" $(a_i, p_i, n_i)$:

* **Anchor ($a_i$):** A reference image of a specific person.
* **Positive ($p_i$):** A different image of the same person as the anchor.
* **Negative ($n_i$):** An image of a different person.

### The Triplet Loss Function

The model is optimized by minimizing the **Triplet Loss**:

$$L(a_i, p_i, n_i) = \max(\|E(a_i) - E(p_i)\|_2 - \|E(a_i) - E(n_i)\|_2 + \text{margin}, 0)$$

The term $\|E(a_i) - E(p_i)\|_2 - \|E(a_i) - E(n_i)\|_2$ forces the anchor-positive distance to be smaller than the anchor-negative distance.

### The Margin

The **margin** represents a threshold of "sufficient" separation. Once the negative image is further from the anchor than the positive image by at least the margin value, the loss becomes zero. This allows the model to prioritize difficult samples rather than over-optimizing data points that are already well-separated.

## Deployment and Identification

Once you have trained this system for long enough, you trust it to make the embeddings of images of the same person close, and the embeddings of images of different people far apart, even for people whose faces were not in the training set.  You then assemble a set of "gallery images" of people you might be interested in recognizing (Maryland police, for example, have the Maryland Image Repository System of drivers' license photographs, mug shots, and other photographs shared by nearby states).  You take each of those faces in your gallery, and calculate their embeddings.  Then, when you have a "probe image," like an image from a security or doorbell camera, you calculate that image's embedding, and extract the $k$ closest embeddings from the gallery images.  Those are your suspects.

## Training a facial recognition system with Contrastive Learning

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torchvision
from torchvision.models import resnet18,ResNet18_Weights
from torchvision.transforms import v2
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
import tqdm
from glob import glob
import random
from torchvision.io import read_image, ImageReadMode

Here I just get a ResNet18 model, and replace its classification layer with an output layer that outputs a 256-dimensional vector.  That will be the size of my embeddings.

In [2]:
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

model = resnet18(weights=ResNet18_Weights.DEFAULT)
num_feats = model.fc.in_features
model.fc = nn.Linear(num_feats, 256)

Here's my dataset.  A lot actually is happening here.  The key thing to understand here is that it returns an *anchor* image, a *positive* image of the same person, and a *negative* image of a different person.

In [3]:
class ContLearnDataset(Dataset):
    def __init__(self, dir, transforms=None):
        self.dir = dir
        self.transforms = transforms
        self.filenames = glob(dir+'/*/*.png')
        self.ids = set([ self.path2id(pth) for pth in self.filenames ])

    def path2id(self, path):
        return path.split('/')[-2]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        anchor = self.filenames[idx]

        id = self.path2id(anchor)
        
        poscands = [fn for fn in self.filenames if id in fn and fn != anchor]
        ind = torch.randint(len(poscands),(1,))[0]
        positive = poscands[ind]
        
        negcands = [fn for fn in self.filenames if id not in fn]
        ind = torch.randint(len(negcands),(1,))[0]
        negative = negcands[ind]

        anchor = read_image(anchor,mode=ImageReadMode.RGB)
        positive = read_image(positive,mode=ImageReadMode.RGB)
        negative = read_image(negative,mode=ImageReadMode.RGB)
        if self.transforms is not None:
            return self.transforms(anchor), self.transforms(positive), self.transforms(negative)
        return anchor, positive, negative

I build my datasets and my dataloader...

In [4]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True)
])
trainingSet = ContLearnDataset('data/faces/training/',transforms=transform)
testingSet = ContLearnDataset('data/faces/testing/',transforms=transform)

dl = DataLoader(trainingSet, batch_size=64, num_workers=0)

I train using TripletMarginLoss, to encourage the embeddings of the same person to be close to each other, and the embeddings of different people to be farther away from each other.

In [5]:
model=model.to('cuda')
EPOCHS = 201

criterion = nn.TripletMarginLoss()
optimizer = optim.Adam(model.parameters(), lr=.001)

if False:
    for epoch in tqdm.tqdm(range(EPOCHS)):
        totalloss=0
        for batch, (a, p, n) in enumerate(dl):
            a,p,n = a.to('cuda'), p.to('cuda'), n.to('cuda')
            aem = model(a)
            pem = model(p)
            nem = model(n)
            loss = criterion(aem, pem, nem)
    
                
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
            totalloss+=loss.item()
        if epoch%20==0:
            print(totalloss)
    torch.save(model, 'fr.pt')

In [7]:
model=torch.load('fr.pt', weights_only=False)
model=model.to('cuda')

Here I test on a testing set, which was not trained on.  `a,p,n` are an anchor, positive, and negative examples from a different dataset of people.  You can see the distance between the anchor and positive example is far smaller than the distances between the anchor and negative, and positive and negative examples.

In [8]:
a,p,n = testingSet[15]
a,p,n = a.reshape((1,3,112,92)), p.reshape((1,3,112,92)), n.reshape((1,3,112,92))
vals=torch.cat((a,p,n),dim=0).to('cuda')
res=model(vals).detach()
print(res.shape)
#print(res)

res[0].shape
print(f'a/p: {nn.functional.mse_loss(res[0],res[1])}')
print(f'a/n: {nn.functional.mse_loss(res[0],res[2])}')
print(f'p/n: {nn.functional.mse_loss(res[1],res[2])}')


torch.Size([3, 256])
a/p: 0.09848816692829132
a/n: 0.32224687933921814
p/n: 0.5249822735786438
